# Outlier Dimensions AnalysisDetect and analyze "massive activations" — dimensions in the residual stream that carry disproportionately large values. Also examines attention sinks and dimension utilization.Recent research (Sun et al., 2024) shows that a few dimensions can dominate the residual stream with magnitudes 100-1000x larger than typical dimensions. Understanding these outliers is critical for quantization and model compression.

## Why This Matters

Outlier dimension analysis identifies the 'massive activations' phenomenon where a small number of residual stream dimensions have disproportionately large values. Understanding attention sinks, dimension utilization, and the impact of removing outliers reveals structural properties of transformer representations.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformercfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])def metric(logits):    return float(logits[-1, 0])

In [ ]:
from irtk.outlier_dimensions import (    detect_outlier_dimensions,    outlier_magnitude_across_layers,    outlier_removal_effect,    attention_sink_analysis,    dimension_utilization_spectrum,)

## Detecting Outlier DimensionsFind dimensions with disproportionately large activations using median absolute deviation (MAD) as a robust outlier measure.

In [ ]:
result = detect_outlier_dimensions(model, tokens, layer=-1)print(f"Median magnitude: {result['median_magnitude']:.6f}")print(f"Max/median ratio: {result['max_to_median_ratio']:.2f}")print(f"Outlier fraction: {result['outlier_ratio']*100:.1f}%")if result['outlier_dims']:    print(f"Top outlier dims:")    for dim, mag in result['outlier_dims'][:5]:        print(f"  Dim {dim}: magnitude = {mag:.6f}")

## Tracking Outliers Across LayersWhere do outlier dimensions emerge? Track their magnitude from embedding through the final layer.

In [ ]:
result = outlier_magnitude_across_layers(model, tokens, target_dims=[0, 5, 10])for d in result['tracked_dims']:    traj = result['magnitude_trajectories'][d]    print(f"Dim {d}: {[f'{t:.6f}' for t in traj]}, "          f"emergence=L{result['emergence_layers'][d]}, "          f"growth={result['growth_rates'][d]:.2f}x")

## Impact of Removing OutliersWhat happens to model output when we clamp outlier dimensions to zero?

In [ ]:
result = outlier_removal_effect(model, tokens, dims_to_clamp=[0, 1, 2])print(f"Clamped dims: {result['clamped_dims']}")print(f"KL divergence: {result['kl_divergence']:.6f}")print(f"Top token prob change: {result['top_token_change']:.6f}")print(f"Prediction changed: {result['prediction_changed']}")print(f"Original entropy: {result['original_entropy']:.4f}")print(f"Clamped entropy: {result['clamped_entropy']:.4f}")

## Attention Sink AnalysisIdentify tokens that receive disproportionate attention (attention sinks) and examine their residual stream properties.

In [ ]:
result = attention_sink_analysis(model, tokens)print(f"Attention received per position:")for p, a in enumerate(result['attention_received']):    marker = " <-- SINK" if p in result['sink_positions'] else ""    print(f"  Position {p}: {a:.4f}{marker}")print(f"Sink positions: {result['sink_positions']}")print(f"Sink/non-sink norm ratio: {result['sink_vs_nonsink_ratio']:.4f}")

## Dimension Utilization SpectrumHow evenly does the model use its d_model dimensions? High Gini coefficient means a few dimensions dominate; low means even utilization.

In [ ]:
result = dimension_utilization_spectrum(model, tokens)print(f"Effective dimensionality: {result['effective_dimensionality']} / 16")print(f"Gini coefficient: {result['gini_coefficient']:.4f}")print(f"Top 10% fraction of variance: {result['top_10_fraction']*100:.1f}%")print(f"Utilization entropy: {result['utilization_entropy']:.4f} (max = {np.log(16):.4f})")print(f"Top 5 dimension variances: {[f'{v:.6f}' for v in result['variance_per_dim'][:5]]}")